In [1]:
import logging
import sys

# Konfiguration für Jupyter erzwingen
logging.basicConfig(
    level=logging.INFO,  # Auf welchen Level wird geloggt 
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout, 
    force=True         
)

# Logger erstellen
logger = logging.getLogger(__name__)

# Testen
logger.debug("Das ist eine Debug-Nachricht (nur für Entwickler).")
logger.info("Das Programm läuft ganz normal weiter.")
logger.warning("Achtung, hier könnte bald ein Problem auftreten!")

09:49:42 - __main__ - INFO - Das Programm läuft ganz normal weiter.
09:49:42 - __main__ - WARNING - Achtung, hier könnte bald ein Problem auftreten!


In [ ]:
import pickle

with open(r"C:\Users\j.beckmann\OneDrive - Reply\Uni Jonas Beckmann\4.Semester\Operations Research\Projekt\Umwelftfreundliche_Routenplanung\models\Berlin\Data\OperationsResearch (1)\berlin_graph_with_accidents.pkl", 'rb') as f:
    berlin_graph_accidents = pickle.load(f)

with open(r"C:\Users\j.beckmann\OneDrive - Reply\Uni Jonas Beckmann\4.Semester\Operations Research\Projekt\Umwelftfreundliche_Routenplanung\models\Berlin\Data\OperationsResearch (1)\berlin_graph_with_nature_new.pkl", 'rb') as f:
    berlin_graph_nature = pickle.load(f)

with open(r"C:\Users\j.beckmann\OneDrive - Reply\Uni Jonas Beckmann\4.Semester\Operations Research\Projekt\Umwelftfreundliche_Routenplanung\models\Berlin\Data\OperationsResearch (1)\berlin_graph_with_population.pkl", 'rb') as f:
    berlin_graph_population = pickle.load(f)



with open(r"C:\Users\j.beckmann\OneDrive - Reply\Uni Jonas Beckmann\4.Semester\Operations Research\Projekt\Umwelftfreundliche_Routenplanung\data\processed\all_data.pkl", 'rb') as f:
    all_data = pickle.load(f)

with open(r"C:\Users\j.beckmann\OneDrive - Reply\Uni Jonas Beckmann\4.Semester\Operations Research\Projekt\Umwelftfreundliche_Routenplanung\models\Berlin\Data\OperationsResearch (1)\berlin_graph_geo_com.pkl", 'rb') as f:
    germany_graph_geo_tun = pickle.load(f)

logger.info("Daten erfolgreich geladen.")

11:38:09 - __main__ - INFO - Daten erfolgreich geladen.


In [14]:
print(f"Accidents_Berlin: {berlin_graph_accidents.keys()}")
print(f"edges: {list(berlin_graph_accidents['edges'].keys())[:5]}")
df_edges = berlin_graph_accidents['edges']
df_nodes = berlin_graph_accidents['nodes']

print(f"edges accidents: \n{df_edges.head()}") #Distanz von den Unfalldaten nehmen 
print(f"nodes accidents: \n{df_nodes.head()}") 

print("----------------------------------------------------------------------------------------")
print(f"Nature_Berlin: {berlin_graph_nature.keys()}")

print(f"Meta: {berlin_graph_nature['meta'].keys()}")
print(f"edges count nature: \n{berlin_graph_nature['edges'].head()}") # dist_to_nature_m  vllt. nature score 
# print("----------------------------------------------------------------------------------------")

# print(f"Population_Berlin: {berlin_graph_population.keys()}")
# print(f"Meta: {berlin_graph_population['meta'].keys()}")
# print(f"edges  population: \n{berlin_graph_population['edges'].head()}") #pop_per_m
# print("----------------------------------------------------------------------------------------")

# print(f"All_Data: {all_data.keys()}")
# print(f"Cosutomer: \n {all_data['customer_locations'].head()}") #customer_id, lat, long, quantity, unit

print("-"*90)
# print(germany_graph_geo_tun.keys())
# print(f"tunnel: {list(germany_graph_geo_tun['tunnel'].values())[:5]}")
# print(f"tunnel: {list(germany_graph_geo_tun['tunnel'].keys())[:5]}")
# print(f"nodes: {germany_graph_geo_tun['nodes'][:5]}")
# print(f"arcs: {germany_graph_geo_tun['arcs'][:5]}")
# print(f"cost: {list(germany_graph_geo_tun['cost'])[:50]}")
# print(f"highway: {list(germany_graph_geo_tun['highway'].values())[:5]}")
# print(f"cords: {germany_graph_geo_tun['coords'][:5]}")



Accidents_Berlin: dict_keys(['edges', 'nodes'])
edges: ['u', 'v', 'distance', 'accidents', 'score']
edges accidents: 
             u            v   distance  accidents     score  weighted_score
0   1822620447  11405216193  13.202038        1.0  0.075746        0.075746
1  11405216193   2845478638  11.135482        0.0  0.000000        0.000000
2   2845478638   2845478635  35.199744        0.0  0.000000        0.000000
3   2845478635   3901927142  79.734382        0.0  0.000000        0.000000
4   3901927142   9929931085  10.779394        0.0  0.000000        0.000000
nodes accidents: 
     node        lat        lon  accidents
0  172539  13.335496  52.565210        0.0
1  172541  13.336417  52.565638        0.0
2  172542  13.337555  52.566000        0.0
3  172543  13.339433  52.566304        0.0
4  172544  13.343230  52.566615        0.0
----------------------------------------------------------------------------------------
Nature_Berlin: dict_keys(['nodes', 'edges', 'meta'])
Meta: di

In [ ]:
# Berechnen des nächstgelegenen Knotens für die Lieferungen

import pandas as pd
from math import radians, sin, cos, sqrt, atan2


def haversine(lat1, lon1, lat2, lon2):
    """
    Berechnet Distanz zwischen zwei GPS-Punkten in Metern
    Haversine-Distanzfunktion
    """

    R = 6371000  # Erdradius in Metern

    lat1, lon1, lat2, lon2 = map(
        radians,
        [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2) ** 2
        + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    )

    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return R * c




# Knotendaten
df_nodes = berlin_graph_accidents["nodes"]

# lat/lon korrigieren
df_nodes = df_nodes.rename(
    columns={
        'lat': 'lon',
        'lon': 'lat'
    }
)

# Kundendaten
df_customers = all_data['customer_locations']



# Kunden auf nächsten Knoten mappen
customer_to_node = {}

for _, customer in df_customers.iterrows():

    customer_lat = customer["Breitengrad"]
    customer_lon = customer["Längengrad"]

    min_distance = float("inf")
    nearest_node = None
    nearest_node_lat = None
    nearest_node_lon = None

    # Alle Knoten durchsuchen
    for _, node in df_nodes.iterrows():

        node_id = node["node"]
        node_lat = node["lat"]
        node_lon = node["lon"]
        
        #Distanz mit Haversine berechnen
        dist = haversine(
            customer_lat,
            customer_lon,
            node_lat,
            node_lon
        )

        # Nächsten Knoten merken
        if dist < min_distance:

            min_distance = dist
            nearest_node = node_id
            nearest_node_lat = node_lat
            nearest_node_lon = node_lon

    # Ergebnis speichern
    customer_to_node[customer["Nr"]] = {

        "customer_name": customer["Ort"],
        "nearest_node": int(nearest_node),

        "node_lat": nearest_node_lat,
        "node_lon": nearest_node_lon,

        "distance_m": round(min_distance, 2)
    }


# Ausgabe zum checken 
for customer_id, info in customer_to_node.items():

    print(f"Kunde {customer_id}")
    print(f"Ort: {info['customer_name']}")
    print(f"Nächster Knoten: {info['nearest_node']}")
    print(f"Knoten Koordinaten: ({info['node_lat']}, {info['node_lon']})")
    print(f"Entfernung: {info['distance_m']} m")
    print("-" * 40)

Kunde 1
Ort: Tankstelle Gütersloh
Nächster Knoten: 1131091766
Knoten Koordinaten: (52.4117548, 13.083971)
Entfernung: 326088.92 m
----------------------------------------
Kunde 2
Ort: Industriegebiet Chemiekonzern Hamburg
Nächster Knoten: 30967210
Knoten Koordinaten: (52.5975692, 13.1462285)
Entfernung: 225550.17 m
----------------------------------------
Kunde 3
Ort: Industriegebiet Berlin
Nächster Knoten: 29533813
Knoten Koordinaten: (52.3924052, 13.3425923)
Entfernung: 2624.56 m
----------------------------------------
Kunde 4
Ort: Industriegebiet Bautzen
Nächster Knoten: 60345055
Knoten Koordinaten: (52.3330987, 13.6388321)
Entfernung: 140455.92 m
----------------------------------------
Kunde 5
Ort: Industriegebiet Bayreuth
Nächster Knoten: 268046355
Knoten Koordinaten: (52.3860657, 13.1535756)
Entfernung: 290096.29 m
----------------------------------------
Kunde 6
Ort: Industriegebiet Erding
Nächster Knoten: 268046355
Knoten Koordinaten: (52.3860657, 13.1535756)
Entfernung: 4614

In [ ]:

def find_nearest_node(lat, lon, df_nodes):
    """
    Findet den nächstgelegenen Knoten zu gegebenen Koordinaten.
    """

    min_distance = float("inf")

    nearest_node = None
    nearest_lat = None
    nearest_lon = None

    # Alle Knoten durchsuchen
    for _, node in df_nodes.iterrows():

        node_id = node["node"]
        node_lat = node["lat"]
        node_lon = node["lon"]

        #hier wieder die Haversine-Distanz berechnen
        dist = haversine(
            lat,
            lon,
            node_lat,
            node_lon
        )
        
        # Nächsten Knoten merken
        if dist < min_distance:

            min_distance = dist

            nearest_node = node_id
            nearest_lat = node_lat
            nearest_lon = node_lon

    return {

        "nearest_node": int(nearest_node),

        "node_lat": nearest_lat,
        "node_lon": nearest_lon,

        "distance_m": round(min_distance, 2)
    }

In [25]:
# Beispielkoordinaten Berlin
lat = 52.534397
lon = 13.207262

nearest_node_result = find_nearest_node(lat, lon, df_nodes)

print(nearest_node_result)

{'nearest_node': 9735117328, 'node_lat': 52.5311542, 'node_lon': 13.2029343, 'distance_m': 464.45}


In [5]:

# RISIKOSCORE-BERECHNUNG
# Risk[e, k] = α·PopDens[e] + β·AccRate[e] + γ·NatRes[e]

import numpy as np
E = list(zip(df_edges['u'], df_edges['v']))
K = [all_data['delivery_routes'].iloc[i]['danger_class'] for i in range(len(all_data['delivery_routes']))]  


#1. Gewichtungsparameter definieren

alpha   = 0.40  # Bevölkerungsdichte-Gewicht
beta    = 0.40  # Unfallrate-Gewicht
gamma   = 0.20  # Naturschutznähe-Gewicht


#2. Daten aus Pickledatei laden


# Population pro Meter Straße
df_pop_edges = berlin_graph_population['edges'][['u', 'v', 'pop_per_meter']].copy()
pop_dict = {(row['u'], row['v']): row['pop_per_meter'] 
            for _, row in df_pop_edges.iterrows()}

logging.debug(f"Population: {len(pop_dict):,} Kanten")
logging.debug(f"Min: {min(pop_dict.values()):.1f}, Max: {max(pop_dict.values()):.1f} Menschen/m")

# Unfallrate pro Kante 
df_acc_edges = berlin_graph_accidents['edges'][['u', 'v', 'score']].copy()
acc_dict = {(row['u'], row['v']): row['score'] 
            for _, row in df_acc_edges.iterrows()}

logging.debug(f"Unfälle: {len(acc_dict):,} Kanten")
logging.debug(f"Min: {min(acc_dict.values()):.0f}, Max: {max(acc_dict.values()):.0f} Unfälle")

# Naturschutznähe 
df_nat_edges = berlin_graph_nature['edges'][['u', 'v', 'dist_to_nature_m']].copy()
nat_dict = {(row['u'], row['v']): row['dist_to_nature_m'] 
            for _, row in df_nat_edges.iterrows()}

logging.debug(f"Naturschutz-Nähe: {len(nat_dict):,} Kanten")
logging.debug(f"Min: {min(nat_dict.values()):.0f} m, Max: {max(nat_dict.values()):.0f} m")


# 3. Normalisiere der scores auf 0 bis 1

# Bevölkerungsdichte 
pop_values = list(pop_dict.values())
pop_min, pop_max = min(pop_values), max(pop_values)
pop_range = pop_max - pop_min if pop_max > pop_min else 1

pop_norm = {
    e: (pop_dict[e] - pop_min) / pop_range 
    for e in pop_dict
}

# Unfallrate 
acc_values = list(acc_dict.values())
acc_min, acc_max = min(acc_values), max(acc_values)
acc_range = acc_max - acc_min if acc_max > acc_min else 1

acc_norm = {
    e: (acc_dict[e] - acc_min) / acc_range 
    for e in acc_dict
}

# Naturschutznähe normalisieren (hier ist näher = höheres Risk) 
nat_values = list(nat_dict.values())
nat_min, nat_max = min(nat_values), max(nat_values)
nat_range = nat_max - nat_min if nat_max > nat_min else 1

# Invers berechnen => näher = Risk höher
nat_norm = {
    e: 1 - ((nat_dict[e] - nat_min) / nat_range)  
    for e in nat_dict
}



#4.Berechne Risk pro Kante

# Basisscore pro Kante (ohne Faktor von Gefahrgutklasse)
edge_risk = {}

for e in E:
    #hier dann die normalisierten Werte oder 0 
    pop_score = pop_norm.get(e, 0)
    acc_score = acc_norm.get(e, 0)
    nat_score = nat_norm.get(e, 0)
    
    # Aggregiere gewichtet
    risk_base = (alpha * pop_score 
                + beta * acc_score 
                + gamma * nat_score)
    # Wert auf [0, 1] begrenzen
    edge_risk[e] = max(0, min(1, risk_base))  

logging.debug(f"{len(edge_risk):,} Basis-Risikoscores berechnet")
logging.debug(f"Min: {min(edge_risk.values()):.4f}, Max: {max(edge_risk.values()):.4f}")




# Gefahrgutklassen-Multiplikatoren (
# Höhere Klassen = höheres Risiko 
hazard_class_factors = {
    '3':        1.0,  # Entzündbare Flüssigkeiten
    '2 (TOC)':  0.8,  # Gase 
    '1.1D':     2.0,  # Sprengstoffe 
    '8':        0.9,  # Ätzende Stoffe
    '9':        0.7,  # Sonstige 
    '6':        1.5,  # Giftige Stoffe 
    '2':        0.8,  # Andere Gase
}


#5. Risk Dictionary befüllen 
Risk_calc = {}

for e in E:
    for k in K:
        # Basis-Risiko mit Gefahrgutklasse-Faktor multiplizieren
        hazard_factor = hazard_class_factors.get(k, 1.0)
        risk_value = edge_risk[e] * hazard_factor
        
        # Normiere auf [0, 1]
        Risk_calc[(e, k)] = max(0, min(1, risk_value))

logging.debug(f"Risk Dictionary : {len(Risk_calc):,} Einträge")
logging.debug(f"{len(E)} Kanten und {len(K)} Klassen")


# 6.Ausgabe: Statistiken & Beispiele

risk_values = list(Risk_calc.values())
logging.debug(f"Minimal:         {min(risk_values):.4f}")
logging.debug(f"Maximum:         {max(risk_values):.4f}")
logging.debug(f"Ø Mittelwert:    {np.mean(risk_values):.4f}")
logging.debug(f"Ø Std.abw:       {np.std(risk_values):.4f}")
logging.debug(f"Median:          {np.median(risk_values):.4f}")

logging.debug(f"TOP 10 RISKANTESTE KANTEN-KLASSE KOMBINATIONEN:")
top_10 = sorted(Risk_calc.items(), key=lambda x: x[1], reverse=True)[:10]
for i, ((e, k), risk) in enumerate(top_10, 1):
    node_from, node_to = e
    logging.debug(f"  {i:2d}. Kante ({node_from:>10} → {node_to:>10}) | Klasse {k:>10}: Risk = {risk:.4f}")

logging.debug(f"TOP 10 SICHERSTE KANTEN-KLASSE KOMBINATIONEN:")
bottom_10 = sorted(Risk_calc.items(), key=lambda x: x[1])[:10]
for i, ((e, k), risk) in enumerate(bottom_10, 1):
    node_from, node_to = e
    logging.debug(f"  {i:2d}. Kante ({node_from:>10} → {node_to:>10}) | Klasse {k:>10}: Risk = {risk:.4f}")



NameError: name 'df_edges' is not defined

In [ ]:

# ALLOW DICTIONARY
# Daten laden


tunnel_data_raw = germany_graph_geo_tun['tunnel']
arcs_data = germany_graph_geo_tun['arcs']

print(f"Kanten: {len(arcs_data):,}")
print(f"Tunnel-Einträge: {len(tunnel_data_raw):,}")


# 1. Tunnel-Mapping erzeugen


tunnel_category_dict = {}

for edge_idx, arc in enumerate(arcs_data):

    u = arc['u']
    v = arc['v']

    e = (u, v)

    tunnel_cat = tunnel_data_raw.get(edge_idx, 'no')

    tunnel_category_dict[e] = tunnel_cat

logger.debug(f"{len(tunnel_category_dict):,} Tunnel-Mappings erstellt")



# OSM Tunneltypen -> ADR Kategorien

tunnel_category_mapping = {

    # Kein Tunnel
    'no': 'A',
    '': 'A',
    None: 'A',

    # Leicht restriktiv
    'building_passage': 'B',

    # Moderat restriktiv
    'covered': 'C',

    # Starke Restriktionen
    'yes': 'D',
    'avalanche_protector': 'D',
}


# ADR Restriktionen

ADR_tunnel_restrictions = {

    'A': {
        k: True for k in K
    },

    'B': {
        k: True for k in K
    },

    'C': {

        '1.1D': False,
        '1.5D': False,

        '2': True,
        '2 (TOC)': True,
        '3': True,
        '4': True,
        '5': True,
        '6': True,
        '8': True,
        '9': True,
    },

    'D': {

        '1.1D': False,
        '1.5D': False,

        '2': True,
        '2 (TOC)': True,
        '3': True,
        '4': True,
        '5': True,

        '6': False,
        '8': True,
        '9': False,
    }
}


# Allow Dictionary erzeugen


Allow_calc  = {}

for e in E:

    raw_tunnel_cat = tunnel_category_dict.get(e, 'no')

    adr_cat = tunnel_category_mapping.get(
        raw_tunnel_cat,
        'A'
    )

    for k in K:

        allowed = ADR_tunnel_restrictions[adr_cat].get(
            k,
            True
        )

        Allow_calc[(e, k)] = 1 if allowed else 0

logger.debug(f"Allow erstellt")
logger.debug(f"Einträge: {len(Allow_calc):,}")


# Beispiele
sample = list(Allow_calc.items())[:10]

for key, value in sample:

    logger.debug(f"{key}: {value}")

Kanten: 3,025,675
Tunnel-Einträge: 3,025,675
✓ 3,025,663 Tunnel-Mappings erstellt
✓ Allow erstellt
Einträge: 566,346

Beispiele:
((1822620447, 11405216193), '3'): 1
((1822620447, 11405216193), '2 (TOC)'): 1
((1822620447, 11405216193), '1.1D'): 1
((1822620447, 11405216193), '8'): 1
((1822620447, 11405216193), '9'): 1
((1822620447, 11405216193), '2'): 1
((11405216193, 2845478638), '3'): 1
((11405216193, 2845478638), '2 (TOC)'): 1
((11405216193, 2845478638), '1.1D'): 1
((11405216193, 2845478638), '8'): 1


In [28]:
unique_tunnel = set(germany_graph_geo_tun['tunnel'].values())

print(unique_tunnel)

{'covered', 'avalanche_protector', 'yes', 'building_passage', 'no'}


In [ ]:
import pulp as pl


# ----- SETS (MENGEN) & PARAMETER --------#

# --- SETS (Mengen) ---
V = [row['type'] for _, row in all_data['vehicles'].iterrows()] 
# Menge der verfügbaren Fahrzeuge  => ['MAN_eTGX', 'Volvo_FH_Electric', 'Mercedes_eActros_600']

L = [row['customer_id'] for _, row in all_data['delivery_routes'].iterrows()]
# [1, 2, 3, 4, 5]
# L =  all_data['delivery_routes']             
# Lieferungen (Standort(lat und long), un-Nummer,substanzname, Klasse, quantity, unit, packaging_type )

K = [all_data['delivery_routes'].iloc[i]['danger_class'] for i in range(len(all_data['delivery_routes']))]         
# Gefahrgutklassen der einzelnen Lieferungen =>['3', '2 (TOC)', '3', '1.1D', '3']

# Richtiges Format für den Solver: [172539, 172541, 172542, ]
N = df_nodes['node'].tolist()
# Alle Knoten des Straßennetzes

# Richtiges Format für den Solver: [(1822620447, 11405216193), (11405216193, 2845478638), ]
E = list(zip(df_edges['u'], df_edges['v']))
# Gerichtete Kanten im Straßennetz 


# --- PARAMETER (Eingabedaten) ---

# Lieferungsdaten (Bezug zu L)
Dem = {row['customer_id']: row['quantity'] for _, row in all_data['delivery_routes'].iterrows()}                
# Gewicht je Lieferung (kg) => Annahme das 1 L = 1 Kg ist 
#{1: 15000, 2: 5000, 3: 21500, 4: 1500, 5: 1000, 6: 19500, 7: 15000, 8: 10000, 9: 2500, 10: 8000}}

Class = {row['customer_id']: row['danger_class'] for _, row in all_data['delivery_routes'].iterrows()} 
# Gefahrgutklasse je Lieferung
#{1: '3', 2: '2 (TOC)', 3: '3', 4: '1.1D', 5: '3', 6: '8', 7: '9', 8: '2', 9: '3', 10: '9'}


depot_node = nearest_node_result["nearest_node"]
lieferungen = all_data['customer_locations']["Nr"]
O_dep = {
    lieferung_id: int(depot_node)
    for lieferung_id in lieferungen
} 

D_dep = {
    customer_id: info["nearest_node"]
    for customer_id, info in customer_to_node.items()
}     # Zielknoten (Destination)            

# Fahrzeugdaten (Bezug zu V) aus (vehicles.csv)
Cap = {row['type']: row['capacity_kg'] for _, row in all_data['vehicles'].iterrows()}    
# Maximale Nutzlast (t) => {'MAN_eTGX': 18, 'Volvo_FH_Electric': 20, 'Mercedes_eActros_600': 22}  

Range = {row['type']: row['range_km'] for _, row in all_data['vehicles'].iterrows()}    
# Akkureichweite => {'MAN_eTGX': 500, 'Volvo_FH_Electric': 470, 'Mercedes_eActros_600': 500} 

FC = {row['type']: row['fixed_cost'] for _, row in all_data['vehicles'].iterrows()}    
# Fixkosten (€/Tag) => {'MAN_eTGX': 180, 'Volvo_FH_Electric': 200, 'Mercedes_eActros_600': 220}

# --- Netzwerk- & Risikodaten (Bezug zu E und K) ---

Dist = {
    (row["u"], row["v"]): float(row["distance"]) / 1000
    for _, row in df_edges.iterrows()
}
# Länge der Kante in km              

#Risiko muss hier berechnet werden mit parametern alpha, beta und gamma
Risk = Risk_calc          # Risikoscore [0, 1] je Kante und Klasse       HIER NOCH MACHEN 
Allow = Allow_calc            # 1 wenn erlaubt, 0 wenn gesperrt (ADR)        HIER NOCH MACHEN 
# Beispiel für eine Sperre: Allow[(('n1', 'ziel_A'), 'klasse_6')] = 0

# Fahrtkosten (Bezug zu V und E)
VC_per_km = all_data['vehicles'].set_index('type')['variable_cost_per_km'].to_dict()       #variable Kosten pro km je Fahrzeugtyp
Energy_per_km = all_data['vehicles'].set_index('type')['energy_kwh_per_km'].to_dict()      # Energieverbrauch pro km je Fahrzeugtyp (kWh/km)

strompreis = 0.35
# Strompreis erstmal fest lassen, weil wir nicht wissen ob Straßentyp an kanten dran steht 
VC = {}
for v in V:
    for e in E:
        # Dist[e] muss vorher definiert sein (z.B. als Dictionary Dist = {e: länge_in_km})
        distanz = Dist[e]
        
        # km-Kosten berechnen
        km_kosten = distanz * VC_per_km[v]
        
        # Energiekosten berechnen
        energie_kosten = distanz * Energy_per_km[v] * strompreis
        
        # Gesamte variable Kosten für dieses Fahrzeug auf dieser Kante
        VC[(v, e)] = km_kosten + energie_kosten

      # Variable Kosten (€) pro Fahrzeug und Kante

# Gewichtungsfaktoren für die Zielfunktion
w1 = 0.5  # Risiko-Gewicht
w2 = 0.5  # Kosten-Gewicht




In [41]:
# print(list(VC.items())[:5])
# print(D_dep)
print(list(Dist.items())[:5])
for d in Dist.items(): 
    if d[1] > 1: 
        print(d)

print(Range)

[((1822620447.0, 11405216193.0), 0.013202037958423606), ((11405216193.0, 2845478638.0), 0.011135482249729984), ((2845478638.0, 2845478635.0), 0.03519974424546483), ((2845478635.0, 3901927142.0), 0.07973438171199551), ((3901927142.0, 9929931085.0), 0.010779393799938105)]
((26983779.0, 2250437920.0), 1.2183073616818416)
((1373882354.0, 1067804966.0), 1.0807456021730664)
((1067804966.0, 1373882354.0), 1.0807456021730664)
((27555347.0, 27555346.0), 1.1910354216682442)
((656472753.0, 8167312849.0), 1.096503605451376)
{'MAN_eTGX': 500, 'Volvo_FH_Electric': 470, 'Mercedes_eActros_600': 500}


In [ ]:
#----------------------  MODELL INITIALISIERUNG--------------------------#
model = pl.LpProblem("Gefahrgut_Routenplanung_MILP", pl.LpMinimize)

Allow = {}

#---------------------- ENTSCHEIDUNGSVARIABLEN--------------------------#

# f[l, e]: Binär - 1, wenn Lieferung l über Kante e transportiert wird
f = pl.LpVariable.dicts("Fluss", [(l, e) for l in L for e in E], cat='Binary')

# y[l, v]: Binär - 1, wenn Lieferung l dem Fahrzeug v zugewiesen wird
y = pl.LpVariable.dicts("Zuweisung", [(l, v) for l in L for v in V], cat='Binary')

# z[v]: Binär - 1, wenn Fahrzeug v heute eingesetzt wird (Aktivierung)
z = pl.LpVariable.dicts("Aktivierung", [v for v in V], cat='Binary')

# u[l, n]: Stetig - MTZ-Hilfsvariable für die Besuchsreihenfolge (Subtour-Eliminierung)
u = pl.LpVariable.dicts("MTZ", [(l, n) for l in L for n in N], lowBound=0, upBound=len(N), cat='Continuous')



#----------------  ZIELFUNKTION (Normiert & Gewichtet)-------------------#

# Berechnung der theoretischen Maxima für Skalierung [0, 1]
Risk_max = max(Risk[(e, k)] for e in E for k in K)
Risk_sum_max = Risk_max * len(L) * len(E) 

Cost_max = (sum(FC[v] for v in V) 
          + sum(max(VC[(v, e)] for v in V) for e in E) * len(L))

# Einzelterme aufbauen
risk_term = pl.lpSum(Risk[(e, Class[l])] * f[(l, e)] for l in L for e in E)
fixed_cost_term = pl.lpSum(FC[v] * z[v] for v in V)
variable_cost_term = pl.lpSum(
    VC[(v, e)] * y[(l, v)] * (1 / len(V)) 
    for l in L for v in V for e in E 
    if Allow.get((e, Class[l]), 1) == 1
)

# Dem Modell hinzufügen
model += (
    (w1 / Risk_sum_max) * risk_term + 
    (w2 / Cost_max) * (fixed_cost_term + variable_cost_term)
), "Minimiere_Risiko_und_Kosten_normiert"



#--------------- NEBENBEDINGUNGEN (Constraints) --------------------------#


# --- C1: Transportpflicht ---
# Jede Lieferung muss exakt einem Fahrzeug zugewiesen werden.
for l in L:
    model += pl.lpSum(y[(l, v)] for v in V) == 1, f"C1_Transportpflicht_{l}"

# --- C2: Kapazitätsbedingung ---
# Gesamtgewicht der zugewiesenen Lieferungen darf die LKW-Nutzlast nicht überschreiten.
for v in V:
    model += pl.lpSum(Dem[l] * y[(l, v)] for l in L) <= Cap[v] * z[v], f"C2_Kapazitaet_{v}"

# --- C3: Aktivierungsbedingung ---
# Fahrzeug v muss als aktiv markiert sein (z[v]=1), wenn ihm eine Lieferung zugewiesen wird.
for l in L:
    for v in V:
        model += z[v] >= y[(l, v)], f"C3_Aktivierung_{l}_{v}"

# --- C4: Gefahrgutrestriktion ---
# Eine Kante darf von einer Lieferung nicht befahren werden, wenn sie gesperrt ist.
# (Es werden aus Performance-Gründen nur Constraints für gesperrte Kanten erzeugt).
for l in L:
    for e in E:
        if Allow.get((e, Class[l]), 1) == 0:
            model += f[(l, e)] <= 0, f"C4_Gefahrgut_Sperre_{l}_{e[0]}_{e[1]}"

# --- C5: Flusserhaltung (Network Flow Conservation) ---
# Garantiert eine zusammenhängende Route vom Start- zum Zielknoten.
in_edges = {n: [e for e in E if e[1] == n] for n in N}
out_edges = {n: [e for e in E if e[0] == n] for n in N}

for l in L:
    for n in N:
        inflow = pl.lpSum(f[(l, e)] for e in in_edges[n])
        outflow = pl.lpSum(f[(l, e)] for e in out_edges[n])

        if n == O_dep[l]:      # Startknoten (Quelle)
            model += outflow - inflow == 1, f"C5_Flow_Start_{l}_{n}"
        elif n == D_dep[l]:    # Zielknoten (Senke)
            model += inflow - outflow == 1, f"C5_Flow_Ziel_{l}_{n}"
        else:                  # Durchgangsknoten
            model += inflow == outflow, f"C5_Flow_Trans_{l}_{n}"

# --- C6: Subtour-Eliminierung (Miller-Tucker-Zemlin) ---
# Verhindert, dass der Solver isolierte, physikalisch unvollständige Kreise (Schleifen) bildet.
M_mtz = len(N)  # Engste mathematisch korrekte Schranke
for l in L:
    for (i, j) in E:
        if j != O_dep[l]:  # Startknoten hat keinen logischen Vorgänger auf der Route
            model += u[(l, i)] - u[(l, j)] + M_mtz * f[(l, (i, j))] <= M_mtz - 1, f"C6_MTZ_{l}_{i}_{j}"

# --- C7: Akkureichweite (E-Truck spezifisch) ---
# Gesamtdistanz der geplanten Route darf die Reichweite des zugewiesenen LKWs nicht überschreiten.
for l in L:
    total_dist = pl.lpSum(Dist[e] * f[(l, e)] for e in E)
    max_range = pl.lpSum(Range[v] * y[(l, v)] for v in V)
    model += total_dist <= max_range, f"C7_Reichweite_{l}"



# --------------------- MODELL LÖSEN --------------------------------#

solver = pl.PULP_CBC_CMD(msg=1, timeLimit=60)
status = model.solve(solver)
print(f"Status: {pl.LpStatus[status]}")

Status: Infeasible


In [45]:
print("\n" + "="*80)
print("CONSTRAINT ANALYSE")
print("="*80)

for name, constraint in model.constraints.items():
  
    lhs_value = constraint.value()

    
    # Prüfe Verletzung
    violated = False

    # <=
    if constraint.sense == -1:
        # if lhs_value > 0:
        #     violated = True
        if lhs_value > 1e-4: 
            violated = True

    # >=
    elif constraint.sense == 1:
        if lhs_value < 0:
            violated = True
            

    # ==
    elif constraint.sense == 0:
        if abs(lhs_value) > 1e-6:
            violated = True

    if violated:
        print(f"\nConstraint: {name}")
        print(f"Value: {lhs_value}")
        print(f"Sense: {constraint.sense}")
        print(f"RHS: {-constraint.constant}")

        print(">>> VERLETZT <<<")


CONSTRAINT ANALYSE

Constraint: C5_Flow_Ziel_1_1131091766
Value: -1.0
Sense: 0
RHS: 1.0
>>> VERLETZT <<<

Constraint: C5_Flow_Start_1_9735117328
Value: -1.0
Sense: 0
RHS: 1.0
>>> VERLETZT <<<

Constraint: C5_Flow_Trans_2_281459207
Value: -1.0
Sense: 0
RHS: -0.0
>>> VERLETZT <<<

Constraint: C5_Flow_Start_2_9735117328
Value: -1.0
Sense: 0
RHS: 1.0
>>> VERLETZT <<<

Constraint: C5_Flow_Trans_3_8707538941
Value: -1.0
Sense: 0
RHS: -0.0
>>> VERLETZT <<<

Constraint: C5_Flow_Start_3_9735117328
Value: -1.0
Sense: 0
RHS: 1.0
>>> VERLETZT <<<

Constraint: C5_Flow_Ziel_4_60345055
Value: -1.0
Sense: 0
RHS: 1.0
>>> VERLETZT <<<

Constraint: C5_Flow_Start_4_9735117328
Value: -1.0
Sense: 0
RHS: 1.0
>>> VERLETZT <<<

Constraint: C5_Flow_Trans_5_484349
Value: -1.0
Sense: 0
RHS: -0.0
>>> VERLETZT <<<

Constraint: C5_Flow_Start_5_9735117328
Value: -1.0
Sense: 0
RHS: 1.0
>>> VERLETZT <<<

Constraint: C5_Flow_Trans_6_484349
Value: -1.0
Sense: 0
RHS: -0.0
>>> VERLETZT <<<

Constraint: C5_Flow_Start_6_973

In [ ]:
# Das dient zum Prüfen, ob die Graphen zusammenhängen 

import networkx as nx

print("\n" + "="*80)
print("GRAPHEN-STRUKTUR-ANALYSE")
print("="*80)

# 1. Graphen aus deinen MILP-Daten (E) aufbauen
G = nx.DiGraph()
G.add_edges_from(E)

# Depot checken
depot = 9735117328  # Dein Startknoten aus dem Log
print(f"--- DEPOT-CHECK ({depot}) ---")
if G.has_node(depot):
    print(f"Existiert im Graphen: JA")
    print(f"Ausgehende Kanten (Outflow möglich): {G.out_degree(depot)}")
    print(f"Eingehende Kanten: {G.in_degree(depot)}")
else:
    print(f"ACHTUNG: Depot existiert GAR NICHT in der Kantenliste E!")

print("\n--- ROUTEN-CHECK ---")
# Für jede Lieferung prüfen, ob ein physikalischer Weg existiert
for l in L:
    ziel = D_dep[l]
    
    if not G.has_node(ziel):
        print(f"Lieferung {l}: Zielknoten {ziel} fehlt im Graphen!")
        continue
        
    try:
        # Prüft, ob es IRGENDEINEN Weg von A nach B gibt
        has_path = nx.has_path(G, depot, ziel)
        if has_path:
            # Wie lang wäre der kürzeste Weg (nur in Anzahl Kanten)?
            path_length = nx.shortest_path_length(G, depot, ziel)
            print(f"Lieferung {l}: Pfad existiert! ({path_length} Kanten)")
        else:
            print(f"Lieferung {l}: KEIN PFAD MÖGLICH (Graphen-Insel!)")
    except nx.NetworkXNoPath:
        print(f"Lieferung {l}: KEIN PFAD MÖGLICH (NetworkXNoPath Exception)")


GRAPHEN-STRUKTUR-ANALYSE
--- DEPOT-CHECK (9735117328) ---
Existiert im Graphen: JA
Ausgehende Kanten (Outflow möglich): 1
Eingehende Kanten: 1

--- ROUTEN-CHECK ---
Lieferung 1: Pfad existiert! (654 Kanten)
Lieferung 2: Pfad existiert! (320 Kanten)
Lieferung 3: Pfad existiert! (972 Kanten)
Lieferung 4: KEIN PFAD MÖGLICH (Graphen-Insel!)
Lieferung 5: KEIN PFAD MÖGLICH (Graphen-Insel!)
Lieferung 6: KEIN PFAD MÖGLICH (Graphen-Insel!)
Lieferung 7: KEIN PFAD MÖGLICH (Graphen-Insel!)
Lieferung 8: KEIN PFAD MÖGLICH (Graphen-Insel!)
Lieferung 9: Pfad existiert! (654 Kanten)
Lieferung 10: KEIN PFAD MÖGLICH (Graphen-Insel!)
